# grayscale correlation
projecting down the results fromm 3 to 1 dimension using PCA. Then correlating the result with the individual input variables.

1. load model
2. load dataset
3. predict results using NOFE and other methods
4. project to grayscale
5. evaluate correlation

The methods compared are NOFE, PCA, t-SNE, UMAP, PHATE, SpatialPCA and Concord: NOFE and all six
baselines produce a 3-D embedding per node, which step 4 reduces to a single grayscale value per
node.

---
### Utilities

In [ ]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('NUMBA_NUM_THREADS', '1')
import sys
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path


project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

sys.path.append(f"{project_root}src/")
# Import utilities
from utilities import KernelNN, KernelNN_subgraph, GraphLMDBReader, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

sys.path.append(f"{project_root}/ERA5/scripts/")
from plot_utilities import *
from utilities_era5 import ERA5_Graph_Generator as Data_Generator
from utilities_era5 import ERA5_Graph_Generator_perSampleNorm as Data_Generator_old
data_path = project_root / 'ERA5' / 'data'

# Device setup
device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import subprocess

import phate

def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'ERA5' / 'trained_models' / model_type / f'{id}.pth'
    config_path = project_root / 'ERA5' / 'trained_models' / model_type / 'logs' / 'configs' / f'{id}.json' 
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    return model, config

# ---------------------------------------------------------------------------
# Method list + baseline backends.
#
# UMAP, Concord and SpatialPCA only exist in their own conda envs, so they are invoked
# as subprocesses. Buffer ids are per-process: with fixed names two concurrently running
# notebooks race on the same .npy/.csv files.
METHOD_NAMES = ['nofe', 'pca', 'tsne', 'umap', 'phate', 'spatialpca', 'concord']
BASELINE_NAMES = METHOD_NAMES[1:]

_spatialpca_proc = None
_spatialpca_dir = f'{project_root}/ERA5/spatialpca'
_spatialpca_call_id = f'spatialpca_buffer_{os.getpid()}'


def _spatialpca_server():
    """Start the persistent SpatialPCA R server once; return the process.

    Library load (SpatialPCA pulls in Seurat, SPARK, Rcpp) costs ~tens of seconds, so
    the server loads it once and then answers one request per stdin line.
    """
    global _spatialpca_proc
    if _spatialpca_proc is not None and _spatialpca_proc.poll() is None:
        return _spatialpca_proc
    _spatialpca_proc = subprocess.Popen(
        ['conda', 'run', '-n', 'spatialpca', '--no-capture-output',
         'Rscript', 'execute_spatialpca_server.R'],
        cwd=_spatialpca_dir, stdin=subprocess.PIPE, stdout=subprocess.PIPE,
        stderr=subprocess.PIPE, text=True, bufsize=1,
        # R searches the *user* library (~/Library/R/<arch>/<ver>/library) before the
        # conda env's own library. CRAN-built packages there link the CRAN R framework
        # rather than conda's libR, and SpatialPCA's Rcpp .so then segfaults at dyn.load.
        env=dict(os.environ, R_LIBS_USER='/nonexistent'),
    )
    ready = _spatialpca_proc.stdout.readline().strip()
    if ready != 'READY':
        raise RuntimeError(
            f'spatialPCA server failed to start: {ready!r}\n'
            f'{_spatialpca_proc.stderr.read()}'
        )
    return _spatialpca_proc


def spatialpca_embed(X, coords, n_components=3):
    """SpatialPCA embedding of X (samples x features) located at coords (samples x d)."""
    proc = _spatialpca_server()
    np.savetxt(f'{_spatialpca_dir}/{_spatialpca_call_id}_X.csv', X, delimiter=',')
    np.savetxt(f'{_spatialpca_dir}/{_spatialpca_call_id}_coords.csv', coords, delimiter=',')

    proc.stdin.write(f'{_spatialpca_call_id} {n_components}\n')
    proc.stdin.flush()
    response = proc.stdout.readline().strip()

    os.remove(f'{_spatialpca_dir}/{_spatialpca_call_id}_X.csv')
    os.remove(f'{_spatialpca_dir}/{_spatialpca_call_id}_coords.csv')

    if response != 'OK':
        raise RuntimeError(f'spatialPCA failed for {_spatialpca_call_id}: {response}')

    Y = np.loadtxt(f'{_spatialpca_dir}/res_{_spatialpca_call_id}.csv',
                   delimiter=',', skiprows=1)
    os.remove(f'{_spatialpca_dir}/res_{_spatialpca_call_id}.csv')
    return Y


def spatialpca_shutdown():
    """Stop the R server (call after the last SpatialPCA embedding)."""
    global _spatialpca_proc
    if _spatialpca_proc is not None and _spatialpca_proc.poll() is None:
        _spatialpca_proc.stdin.write('QUIT\n')
        _spatialpca_proc.stdin.flush()
        _spatialpca_proc.wait(timeout=10)
    _spatialpca_proc = None


def external_embed(env, workdir, script, buffer_id, X):
    """Run `script` under conda env `env`: it reads <buffer_id>.npy, writes res_<buffer_id>.npy."""
    np.save(f'{workdir}/{buffer_id}.npy', X)
    subprocess.run(f'cd {workdir} && conda run -n {env} python {script} {buffer_id}', shell=True)
    Y = np.load(f'{workdir}/res_{buffer_id}.npy')
    os.remove(f'{workdir}/{buffer_id}.npy')
    os.remove(f'{workdir}/res_{buffer_id}.npy')
    return Y


def predict_baselines(sample, n=3):
    """{baseline: array(n_nodes, n)} for the six baselines, in METHOD_NAMES order."""
    X = sample.normed_values.detach().cpu().numpy()
    coords = sample.normed_coords.detach().cpu().numpy()
    preds = {}

    print('PCA ...', end='\r')
    preds['pca'] = PCA(n_components=n).fit_transform(X)
    print('PCA\t- done')

    print('t-SNE ...', end='\r')
    preds['tsne'] = TSNE(
        n_components=n, perplexity=30, learning_rate='auto', random_state=42
    ).fit_transform(X)
    print('t-SNE\t- done')

    print('UMAP ...', end='\r')
    preds['umap'] = external_embed(
        'umap', f'{project_root}/ERA5/umap', 'execute_umap.py',
        f'gray_umap_buffer_{os.getpid()}', X,
    )
    print('UMAP\t- done')

    print('PHATE ...', end='\r')
    preds['phate'] = phate.PHATE(
        n_components=n, knn=15, mds_solver='smacof', random_state=42, verbose=False
    ).fit_transform(X)
    print('PHATE\t- done')

    print('spatialPCA ...', end='\r')
    preds['spatialpca'] = spatialpca_embed(X, coords, n_components=n)
    print('spatialPCA\t- done')

    print('Concord ...', end='\r')
    preds['concord'] = external_embed(
        'concord', f'{project_root}/ERA5/concord', 'execute_concord.py',
        f'gray_concord_buffer_{os.getpid()}', X,
    )
    print('Concord\t- done')

    return preds


def predict_all(sample, n=3):
    """{method: array(n_nodes, n)} for every entry of METHOD_NAMES, NOFE included."""
    return {'nofe': model(sample).detach().cpu().numpy(), **predict_baselines(sample, n=n)}


----
### load model + dataset 

In [ ]:
#model, config = load_model(id='model5_grayscale_2026-04-28_t12-23-06' , model_type='regular')
model, config = load_model(id='test1_model5_median_tkernel_2026-04-22_t21-22-37' , model_type='regular')

In [ ]:
lat_min = config["lat_min"]
lat_max = config["lat_max"]
lon_min = config["lon_min"]
lon_max = config["lon_max"]
#N = config["N"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
train_date_start_end = config["train_date_start_end"]
test_date_start_end = config["test_date_start_end"]
train_samples_per_date = config["train_samples_per_date"]
test_samples_per_date = config["test_samples_per_date"]
feature_mode = config["feature_mode"]
vars = config["vars"]
output_dim = config["out_width"]

In [ ]:
date = "2019-01-10"

bbox = lon_min, lat_min, lon_max, lat_max

#Generator = Data_Generator_old(data_path)
Generator = Data_Generator(data_path)
Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
print(f'total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')

N=10000
print('generate data')
graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode)
sample = graph.to(device)

locations = pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]})
X = sample.normed_values.detach().cpu().numpy()

---
## Get Predictions

In [ ]:
preds = predict_all(sample, n=output_dim)


---
## Grayscale Projection

In [ ]:
grayscale = {
    method: PCA(n_components=1).fit_transform(preds[method]).squeeze()
    for method in METHOD_NAMES
}


---
## Correlate

In [ ]:
correlations = {
    method: np.array([
        np.abs(np.corrcoef(grayscale[method], X[:, i])[0, 1])
        for i in range(X.shape[1])
    ])
    for method in METHOD_NAMES
}


In [ ]:
corrs = pd.DataFrame(correlations, index=config['vars'])
print(corrs)
print()
print('Mean Values:')
for method in METHOD_NAMES:
    print(f'{method}:\t{corrs[method].mean():.2f}')


---
# Run on full scale datset

In [ ]:
config['dataset_id']

In [ ]:
dates = pd.date_range("2019-01-01", "2019-01-31").strftime("%Y-%m-%d").tolist()

#Generator = Data_Generator_old(data_path)
Generator = Data_Generator(data_path)

bbox = lon_min, lat_min, lon_max, lat_max


print('generate data')

graphs=[]
locations_list=[]
X_list=[]

for date in dates:
    N=random.randint(1000,10000)
    Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
    print(f'total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')
    graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode)
    sample = graph.to(device)
    graphs.append(sample)
    locations_list.append(pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]}))
    X_list.append(sample.normed_values.detach().cpu().numpy())

In [ ]:
#dataset = GraphLMDBReader(f'{project_root}/ERA5/datasets/regular/{config["dataset_id"]}_test.lmdb')
dataset = graphs
print(f'loaded {len(dataset)} samples')

In [ ]:
corr_res = {method: [] for method in METHOD_NAMES}

# One pass over the dataset: each sample is mounted once and every method is embedded
# once, then reduced to grayscale and correlated against that sample's input channels.
for s in range(len(dataset)):
    print(f'{s+1}/{len(dataset)}', end='\r')

    sample = dataset[s].to(device='mps')
    X = sample.normed_values.detach().cpu().numpy()

    preds_s = predict_all(sample, n=output_dim)

    for method in METHOD_NAMES:
        grayscale_s = PCA(n_components=1).fit_transform(preds_s[method]).squeeze()
        corr_res[method].append(np.array([
            np.abs(np.corrcoef(grayscale_s, X[:, i])[0, 1])
            for i in range(X.shape[1])
        ]))

# last SpatialPCA call is done; the R server would otherwise outlive the notebook
spatialpca_shutdown()

corr_res = {method: np.vstack(corr_res[method]) for method in METHOD_NAMES}

for method in METHOD_NAMES:
    print(f'{method}:\t{corr_res[method].mean():.3f}')


In [ ]:
corrs = pd.DataFrame(
    {method: corr_res[method].mean(axis=0) for method in METHOD_NAMES},
    index=config['vars'],
)

print('Mean correlation per variable:')
print(corrs)

print('\nMean correlation coefficient:\n')
for method in METHOD_NAMES:
    print(f'{method}:\t{corr_res[method].mean():.3f}')


In [ ]:
for method in METHOD_NAMES:
    print(f'{method}: ', end='')
    for value in corrs[method]:
        print(f'{round(value, 3)}, & ', end='')
    print()


In [ ]:
corrs.shape

In [ ]:
for i in range(corrs.shape[0]):
    print(' & '.join(f'{round(corrs[method].iloc[i], 3)}' for method in METHOD_NAMES))
